# TikTok claims and opinions
## Data inspection and engagement analysis
**Hernán Licastro | Educational portfolio project**

This case study uses the synthetic dataset supplied with the Google Advanced Data Analytics course activity. The company scenario and stakeholders are fictional. It is not employment at, or an analysis of production data from, TikTok.

**Business question:** What data quality issues and preliminary engagement patterns should the team understand before developing a claims classification model?

**Scope:** Import, inspect, summarize and organize the data using PACE. No model, formal hypothesis test or causal inference is delivered. A claim is a statement presented as factual; the course narrows this to unsourced or unverified information. It is not synonymous with misinformation, a policy violation or publication approval.

## Plan
Review the proposal, dictionary and stakeholder requests. Treat one row as one video. Separate identifiers, labels, text, author attributes and engagement counts. Preserve the raw CSV and document all exclusions and denominators. The supplied dictionary describes 19,383 records; verify this rather than assuming it is accurate.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
DATA_PATH = Path("../data/tiktok_dataset.csv") if Path("../data/tiktok_dataset.csv").exists() else Path("data/tiktok_dataset.csv")
data = pd.read_csv(DATA_PATH)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | NumPy {np.__version__}")
print(f"Rows: {data.shape[0]:,}; columns: {data.shape[1]}")
data.head(10)

Python 3.12.14 | pandas 2.2.3 | NumPy 2.3.5
Rows: 19,382; columns: 12


,#,claim_status,video_id,video_duration_sec,video_transcription_text,verified_status,author_ban_status,video_view_count,video_like_count,video_share_count,video_download_count,video_comment_count
0,1,claim,7017666017,59,someone shared with me that drone deliveries a...,not verified,under review,343296.0,19425.0,241.0,1.0,0.0
1,2,claim,4014381136,32,someone shared with me that there are more mic...,not verified,active,140877.0,77355.0,19034.0,1161.0,684.0
2,3,claim,9859838091,31,someone shared with me that american industria...,not verified,active,902185.0,97690.0,2858.0,833.0,329.0
3,4,claim,1866847991,25,someone shared with me that the metro of st. p...,not verified,active,437506.0,239954.0,34812.0,1234.0,584.0
4,5,claim,7105231098,19,someone shared with me that the number of busi...,not verified,active,56167.0,34987.0,4110.0,547.0,152.0
5,6,claim,8972200955,35,someone shared with me that gross domestic pro...,not verified,under review,336647.0,175546.0,62303.0,4293.0,1857.0
6,7,claim,4958886992,16,someone shared with me that elvis presley has ...,not verified,active,750345.0,486192.0,193911.0,8616.0,5446.0
7,8,claim,2270982263,41,someone shared with me that the best selling s...,not verified,active,547532.0,1072.0,50.0,22.0,11.0
8,9,claim,5235769692,50,someone shared with me that about half of the ...,not verified,active,24819.0,10160.0,1050.0,53.0,27.0
9,10,claim,4660861094,45,someone shared with me that it would take a 50...,verified,active,931587.0,171051.0,67739.0,4104.0,2540.0


## Analyze
### Structure and data quality
The first ten records are claims; this preview does not establish the overall class distribution. The unnamed display index is separate from `#`. Identifiers are not continuous explanatory measurements.

In [2]:
data.info()
column_summary = pd.DataFrame({"dtype": data.dtypes.astype(str), "non_null": data.notna().sum(), "missing": data.isna().sum(), "missing_pct": data.isna().mean().mul(100)})
column_summary

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19382 entries, 0 to 19381
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   #                         19382 non-null  int64  
 1   claim_status              19084 non-null  object 
 2   video_id                  19382 non-null  int64  
 3   video_duration_sec        19382 non-null  int64  
 4   video_transcription_text  19084 non-null  object 
 5   verified_status           19382 non-null  object 
 6   author_ban_status         19382 non-null  object 
 7   video_view_count          19084 non-null  float64
 8   video_like_count          19084 non-null  float64
 9   video_share_count         19084 non-null  float64
 10  video_download_count      19084 non-null  float64
 11  video_comment_count       19084 non-null  float64
dtypes: float64(5), int64(3), object(4)
memory usage: 1.8+ MB


,dtype,non_null,missing,missing_pct
#,int64,19382,0,0.000000
claim_status,object,19084,298,1.537509
video_id,int64,19382,0,0.000000
video_duration_sec,int64,19382,0,0.000000
video_transcription_text,object,19084,298,1.537509
verified_status,object,19382,0,0.000000
author_ban_status,object,19382,0,0.000000
video_view_count,float64,19084,298,1.537509
video_like_count,float64,19084,298,1.537509
video_share_count,float64,19084,298,1.537509


In [3]:
original_columns = [c for c in data.columns if c not in ["likes_per_view", "comments_per_view", "shares_per_view"]]
raw = data[original_columns]
missing_columns = raw.columns[raw.isna().any()].tolist()
quality_checks = pd.Series({
    "rows": len(raw),
    "original_columns": len(raw.columns),
    "duplicate_rows": int(raw.duplicated().sum()),
    "duplicate_video_ids": int(raw["video_id"].duplicated().sum()),
    "rows_with_any_missing": int(raw.isna().any(axis=1).sum()),
    "rows_missing_all_seven_fields": int(raw[missing_columns].isna().all(axis=1).sum()),
    "nonpositive_known_views": int(raw["video_view_count"].le(0).sum())
})
quality_checks

rows                             19382
original_columns                    12
duplicate_rows                       0
duplicate_video_ids                  0
rows_with_any_missing              298
rows_missing_all_seven_fields      298
nonpositive_known_views              0
dtype: int64

The CSV contains **19,382 rows and 12 original columns**, one fewer row than the dictionary states. There are no duplicate rows or repeated video IDs. The **same 298 rows** lack the label, transcription and all five engagement counts; the remaining 19,084 are complete across the original fields. The reason for missingness is unknown. No records are removed or imputed.

The lab environment reported 3 integer, 5 float and 4 object columns. Pandas versions may infer string columns differently; the generated summary is authoritative for each execution. Floats can hold missing count values; this does not make the underlying counts continuous.

In [4]:
data.describe()

,#,video_id,video_duration_sec,video_view_count,video_like_count,video_share_count,video_download_count,video_comment_count
count,19382.000000,1.938200e+04,19382.000000,19084.000000,19084.000000,19084.000000,19084.000000,19084.000000
mean,9691.500000,5.627454e+09,32.421732,254708.558688,84304.636030,16735.248323,1049.429627,349.312146
std,5595.245794,2.536440e+09,16.229967,322893.280814,133420.546814,32036.174350,2004.299894,799.638865
min,1.000000,1.234959e+09,5.000000,20.000000,0.000000,0.000000,0.000000,0.000000
25%,4846.250000,3.430417e+09,18.000000,4942.500000,810.750000,115.000000,7.000000,1.000000
50%,9691.500000,5.618664e+09,32.000000,9954.500000,3403.500000,717.000000,46.000000,9.000000
75%,14536.750000,7.843960e+09,47.000000,504327.000000,125020.000000,18222.000000,1156.250000,292.000000
max,19382.000000,9.999873e+09,60.000000,999817.000000,657830.000000,256130.000000,14994.000000,9599.000000


In [5]:
data[["video_duration_sec", "video_view_count"]].agg(["min", "max", "mean", "median"])

,video_duration_sec,video_view_count
min,5.000000,20.000000
max,60.000000,999817.000000
mean,32.421732,254708.558688
median,32.000000,9954.500000


Durations range from 5 to 60 seconds. Views range from 20 to 999,817. Engagement means exceed medians substantially in the pooled data. These patterns warrant distribution and subgroup exploration, not automatic removal of high values. Duration and counts are ratio-scale quantities, rather than interval-scale measurements in the strict statistical sense.

### Classification balance
Report percentages over both all records and known labels. Missing labels are not a third substantive video category.

In [6]:
counts = data["claim_status"].value_counts(dropna=False)
classification_summary = pd.DataFrame({"videos": counts, "pct_all_records": counts / len(data) * 100})
classification_summary["pct_known_labels"] = counts / data["claim_status"].notna().sum() * 100
classification_summary.loc[classification_summary.index.isna(), "pct_known_labels"] = np.nan
classification_summary

,videos,pct_all_records,pct_known_labels
claim_status,,,
claim,9608,49.571768,50.345839
opinion,9476,48.890723,49.654161
NaN,298,1.537509,NaN


Claims represent **50.35%** and opinions **49.65%** of known labels. Over all records, the shares are **49.57% claims, 48.89% opinions and 1.54% missing labels**. The known classes are nearly balanced.

In [7]:
claim_views = data.loc[data["claim_status"] == "claim", "video_view_count"]
opinion_views = data.loc[data["claim_status"] == "opinion", "video_view_count"]
view_summary = pd.DataFrame({"claim": claim_views.agg(["mean", "median"]), "opinion": opinion_views.agg(["mean", "median"])})
print("Mean view ratio:", claim_views.mean() / opinion_views.mean())
view_summary

Mean view ratio: 101.08671469450185


,claim,opinion
mean,501029.452748,4956.43225
median,501555.000000,4953.00000


Claim videos average **501,029.45 views**, compared with **4,956.43** for opinions, a ratio of approximately **101.09**. Within each class, means and medians are close. This does not prove symmetry or absence of outliers. Mixing groups with very different reach helps explain the pooled mean–median gap.

In [8]:
status_counts = data.groupby(["claim_status", "author_ban_status"]).size().rename("videos")
status_summary = status_counts.to_frame()
status_summary["pct_within_claim_status"] = status_counts / status_counts.groupby(level=0).transform("sum") * 100
status_summary

videos  pct_within_claim_status
claim_status author_ban_status                                 
claim        active               6566                68.338884
             banned               1439                14.977102
             under review         1603                16.684013
opinion      active               8817                93.045589
             banned                196                 2.068383
             under review          463                 4.886028

Banned authors account for **14.98% of claim videos** versus **2.07% of opinion videos**. Under-review authors account for **16.68%** versus **4.89%**. These are video counts, not counts of unique authors. Greater exposure or reporting is one possible explanation, but ban reasons and timestamps are unavailable; no causal interpretation is justified.

In [9]:
data.groupby("author_ban_status")["video_share_count"].median()

author_ban_status
active            437.0
banned          14468.0
under review     9444.0
Name: video_share_count, dtype: float64

Median shares are **14,468** for banned authors, **9,444** for authors under review and **437** for active authors. The banned-to-active ratio is approximately 33.11. Category composition and reach may contribute to this difference.

In [10]:
data.groupby("author_ban_status").agg({
    "video_view_count": ["count", "mean", "median"],
    "video_like_count": ["count", "mean", "median"],
    "video_share_count": ["count", "mean", "median"]
})

video_view_count                 ... video_share_count         
                             count           mean  ...              mean   median
author_ban_status                                  ...                           
active                       15383  215927.039524  ...      14111.466164    437.0
banned                        1635  445845.439144  ...      29998.942508  14468.0
under review                  2066  392204.836399  ...      25774.696999   9444.0

[3 rows x 9 columns]

Raw engagement means and medians are highest for banned authors, followed by under-review and active authors. `count` excludes null values; `size` counts rows. These comparisons use available metrics and do not describe the 298 records with missing engagement.

In [11]:
valid_views = data["video_view_count"].where(data["video_view_count"] > 0)
for metric, source in {"likes_per_view": "video_like_count", "comments_per_view": "video_comment_count", "shares_per_view": "video_share_count"}.items():
    data[metric] = data[source] / valid_views
rates = data.groupby(["claim_status", "author_ban_status"]).agg({
    "likes_per_view": ["count", "mean", "median"],
    "comments_per_view": ["count", "mean", "median"],
    "shares_per_view": ["count", "mean", "median"]
})
rates

likes_per_view            ... shares_per_view          
                                        count      mean  ...            mean    median
claim_status author_ban_status                           ...                          
claim        active                      6566  0.329542  ...        0.065456  0.049279
             banned                      1439  0.345071  ...        0.067893  0.051606
             under review                1603  0.327997  ...        0.065733  0.049967
opinion      active                      8817  0.219744  ...        0.043729  0.032405
             banned                       196  0.206868  ...        0.040531  0.030728
             under review                 463  0.226394  ...        0.044472  0.035027

[6 rows x 9 columns]

Claims have higher mean and median interactions per view than opinions within every author status. Mean likes per view range from **0.328–0.345** for claims and **0.207–0.226** for opinions. Mean comments per view are **0.00137–0.00139** versus **0.00043–0.00054**; mean shares per view are **0.0655–0.0679** versus **0.0405–0.0445**.

Banned authors do not consistently have the highest normalized rates within classes. These are unweighted means of per-video ratios, not total interactions divided by total views, and not proportions of unique people. The denominator guard preserves missingness and avoids invalid division.